<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# ============================================================
# ML-08 — PART 1
# METHOD CHOICE AND WHY
# ============================================================

# REASONING:
# I chose Logistic Regression because my lane is a binary decline
# classification and ranking problem.
#
# Logistic Regression is simple and interpretable, and it produces
# probabilities that can be used to rank pages for review.
#
# It gives me a clear ML benchmark before trying more complex models.
# The purpose is not to reward complexity, but to test whether an
# ML model can improve on my Week-4 hand-written baseline.
#
# I will use observable pre-decision signals only.
# I will not use trend_direction, trend_pct, or is_declining_label
# as model features because these are label-derived and would cause
# leakage.

%pip install -q pandas numpy scikit-learn

import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# LOAD STARTER DATASET
# ------------------------------------------------------------

possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv"
]

csv_path = next(
    (p for p in possible_paths if os.path.exists(p)),
    None
)

if csv_path is None:
    csv_url = (
        "https://raw.githubusercontent.com/"
        "Nayab-khalid/FlyRank-AI-Internship/"
        "main/data/raw/content_refresh_anonymized.csv"
    )

    df = pd.read_csv(csv_url)
    print("Loaded dataset from GitHub.")
else:
    df = pd.read_csv(csv_path)
    print("Loaded dataset from local repository.")

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# DEFINE LABEL
# ------------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nLabel distribution:")
display(
    df["is_declining_label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="n")
)

# ------------------------------------------------------------
# SAFE MODEL FEATURES
# ------------------------------------------------------------

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level"
]

# Keep only columns that actually exist.
numeric_features = [
    c for c in numeric_features
    if c in df.columns
]

categorical_features = [
    c for c in categorical_features
    if c in df.columns
]

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

# ------------------------------------------------------------
# MODEL PIPELINE
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

print("\nMethod selected: Logistic Regression")
print("Purpose: interpretable binary classification and ranking.")


Loaded dataset from GitHub.
Dataset shape: (30000, 44)

Label distribution:


,label,n
0,1,16262
1,0,13738



Numeric features:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features:
['content_type', 'main_intent', 'competition_level']

Method selected: Logistic Regression
Purpose: interpretable binary classification and ranking.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# ============================================================
# ML-08 — PART 2
# SPLIT DESIGN
# ============================================================

# REASONING:
# I use a client-grouped train/test split because pages belonging
# to the same client can share characteristics.
#
# Keeping complete clients in either train or test reduces the risk
# that the model learns client-specific patterns and then appears
# to generalize simply because it has already seen the same client.
#
# The Week-4 baseline will use exactly the same test rows.
# This makes the model-versus-baseline comparison fair.
#
# I use an 80/20 grouped split and a fixed random seed so that the
# experiment is reproducible.

from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# PREPARE MODEL DATA
# ------------------------------------------------------------

feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

# ------------------------------------------------------------
# CLIENT-GROUPED SPLIT
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_rows = df.iloc[train_idx].copy()
test_rows = df.iloc[test_idx].copy()

# ------------------------------------------------------------
# VERIFY CLIENT SEPARATION
# ------------------------------------------------------------

train_clients = set(train_rows["client_id"])
test_clients = set(test_rows["client_id"])

overlap = train_clients.intersection(test_clients)

print("=" * 60)
print("GROUPED SPLIT CHECK")
print("=" * 60)

print("Training rows:", len(train_rows))
print("Test rows:", len(test_rows))

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("Client overlap:", len(overlap))

if len(overlap) == 0:
    print("✓ No client appears in both train and test.")
else:
    print("⚠ Client leakage detected.")

print("\nTraining label rate:")
print(y_train.mean())

print("\nTest label rate:")
print(y_test.mean())


GROUPED SPLIT CHECK
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
✓ No client appears in both train and test.

Training label rate:
0.5501111717078492

Test label rate:
0.5109524582184002


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# ============================================================
# ML-08 — PART 3
# TRAIN + COMPARE VS MY BASELINE
# ============================================================

# REASONING:
# I train Logistic Regression on the training clients only.
#
# The model and the Week-4 baseline are evaluated on exactly the
# same held-out test rows.
#
# I use Average Precision as the main ranking metric because the
# goal is to prioritize useful pages for review rather than simply
# classify every page correctly.
#
# I also report Precision@50 because the Week-4 baseline is a
# ranked action queue and the starter workflow uses Precision@50
# as a useful review-prioritization metric.
#
# A higher score for the model is useful only if it represents a
# genuine improvement over the transparent baseline.

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score
)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

model.fit(X_train, y_train)

# Model probabilities
model_prob = model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# RECREATE WEEK-4 BASELINE ON THE SAME TEST ROWS
# ------------------------------------------------------------

baseline = test_rows.copy()

baseline["impressions_90d"] = pd.to_numeric(
    baseline["impressions_90d"],
    errors="coerce"
).fillna(0)

baseline["days_since_last_update"] = pd.to_numeric(
    baseline["days_since_last_update"],
    errors="coerce"
).fillna(0)

# Staleness score
baseline["staleness_score"] = np.select(
    [
        baseline["days_since_last_update"] >= 180,
        baseline["days_since_last_update"] >= 90
    ],
    [
        2,
        1
    ],
    default=0
)

# Visibility score
baseline["visibility_score"] = np.select(
    [
        baseline["impressions_90d"] >= 3000,
        baseline["impressions_90d"] >= 500,
        baseline["impressions_90d"] >= 100
    ],
    [
        3,
        2,
        1
    ],
    default=0
)

baseline["baseline_score"] = (
    baseline["staleness_score"] +
    baseline["visibility_score"]
)

# ------------------------------------------------------------
# NORMALIZE BASELINE SCORE INTO A RANKING VALUE
# ------------------------------------------------------------

baseline_prob = (
    baseline["baseline_score"].astype(float)
)

# Add a tiny deterministic tie-break using impressions.
# This does not use the label.
baseline_prob = (
    baseline_prob
    + baseline["impressions_90d"].rank(method="average")
    / (len(baseline) * 1000000)
)

# ------------------------------------------------------------
# METRIC FUNCTION
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k=50):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(scores))

    order = np.argsort(-scores)[:k]

    return y_true[order].mean()

# ------------------------------------------------------------
# CALCULATE METRICS
# ------------------------------------------------------------

model_ap = average_precision_score(
    y_test,
    model_prob
)

baseline_ap = average_precision_score(
    y_test,
    baseline_prob
)

model_auc = roc_auc_score(
    y_test,
    model_prob
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_prob
)

model_p50 = precision_at_k(
    y_test,
    model_prob,
    50
)

baseline_p50 = precision_at_k(
    y_test,
    baseline_prob,
    50
)

# ------------------------------------------------------------
# COMPARISON TABLE
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "average_precision": [
        baseline_ap,
        model_ap
    ],
    "roc_auc": [
        baseline_auc,
        model_auc
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ]
})

print("=" * 60)
print("MODEL VS BASELINE")
print("=" * 60)

display(comparison)

# ------------------------------------------------------------
# INTERPRETATION
# ------------------------------------------------------------

if model_ap > baseline_ap:
    print(
        "\nObserved result: Logistic Regression has higher "
        "Average Precision than the Week-4 baseline."
    )
elif model_ap < baseline_ap:
    print(
        "\nObserved result: the Week-4 baseline has higher "
        "Average Precision than Logistic Regression."
    )
else:
    print(
        "\nObserved result: both methods have the same "
        "Average Precision."
    )

# ------------------------------------------------------------
# SAVE METRICS RECEIPT
# ------------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

comparison.to_json(
    "work/outputs/model_vs_baseline_metrics.json",
    orient="records",
    indent=2
)

print(
    "\nMetrics saved to "
    "work/outputs/model_vs_baseline_metrics.json"
)


MODEL VS BASELINE


,method,average_precision,roc_auc,precision_at_50
0,Week-4 baseline,0.489069,0.506511,0.3
1,Logistic Regression,0.871540,0.849590,1.0



Observed result: Logistic Regression has higher Average Precision than the Week-4 baseline.

Metrics saved to work/outputs/model_vs_baseline_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# ============================================================
# ML-08 — PART 4
# ERRORS AND INTERPRETATION
# ============================================================

# REASONING:
# I inspect false positives and false negatives instead of relying
# only on the overall metric.
#
# False positives are pages the model ranks highly even though they
# are not in the positive decline class.
#
# False negatives are positive pages that receive relatively low
# model scores.
#
# I also inspect Logistic Regression coefficients to understand which
# features the model leans on.
#
# These observations are treated as measured or directional findings.
# They are not claims about causality or Google's ranking algorithm.

# ------------------------------------------------------------
# CREATE ERROR TABLE
# ------------------------------------------------------------

error_df = test_rows[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "days_since_last_update",
        "trend_direction"
    ]
].copy()

error_df["actual"] = y_test.values
error_df["model_probability"] = model_prob

# ------------------------------------------------------------
# FALSE POSITIVES
# ------------------------------------------------------------

false_positives = (
    error_df[
        (error_df["actual"] == 0)
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(10)
)

# ------------------------------------------------------------
# FALSE NEGATIVES
# ------------------------------------------------------------

false_negatives = (
    error_df[
        (error_df["actual"] == 1)
    ]
    .sort_values(
        "model_probability",
        ascending=True
    )
    .head(10)
)

print("=" * 60)
print("TOP FALSE POSITIVES")
print("=" * 60)

display(false_positives)

print("=" * 60)
print("TOP FALSE NEGATIVES")
print("=" * 60)

display(false_negatives)

# ------------------------------------------------------------
# FEATURE INTERPRETATION
# ------------------------------------------------------------

classifier = model.named_steps["classifier"]
preprocessor_fitted = model.named_steps["preprocessor"]

feature_names = (
    preprocessor_fitted
    .get_feature_names_out()
)

coefficients = classifier.coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

importance = importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("=" * 60)
print("MOST INFLUENTIAL FEATURES")
print("=" * 60)

display(importance.head(15))

print("""
INTERPRETATION:

The error tables show where the model's ranking disagrees with the
observed decline label.

False positives may represent pages that look risky according to
their available historical signals but do not decline in the observed
label period.

False negatives may represent pages whose available features look
healthy even though they are labelled as declining.

The coefficient table provides a directional view of which features
the Logistic Regression model relies on most. These relationships
should be treated as associations in this dataset, not causal effects.
""")


TOP FALSE POSITIVES


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,days_since_last_update,trend_direction,actual,model_probability
3488,content_a0777b0fd936,client_f369cb89fc,13502,3,9,8,stable,0,0.824482
12869,content_5d5653c4eb4f,client_4e07408562,15101,0,1,7,stable,0,0.756543
19321,content_c149dfab5d24,client_f369cb89fc,2346,3,3,20,stable,0,0.745758
10175,content_374e795aab68,client_f369cb89fc,235,2,3,20,stable,0,0.739409
18326,content_459756cca996,client_f369cb89fc,256,0,4,20,stable,0,0.721205
8724,content_15e0e6081fe9,client_f369cb89fc,950,5,7,20,stable,0,0.717122
16052,content_07d062db2b51,client_f369cb89fc,873,2,2,8,up,0,0.716402
4397,content_53a574b8a7da,client_f369cb89fc,1288,0,2,20,stable,0,0.714844
20390,content_28dc7ce0cfbb,client_f369cb89fc,1355,7,7,20,stable,0,0.711881
20736,content_41baf0722ad9,client_8527a891e2,3115,0,4,104,stable,0,0.708937


TOP FALSE NEGATIVES


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,days_since_last_update,trend_direction,actual,model_probability
24849,content_2f002563e9cd,client_e629fa6598,17,0,37,20,down,1,0.137331
27487,content_31c66d071a62,client_8527a891e2,6,0,42,20,down,1,0.159919
23511,content_4de8c62603bf,client_e629fa6598,10,0,1,20,down,1,0.164663
7769,content_9524a5115a30,client_e629fa6598,15,0,1,20,down,1,0.170926
7042,content_b3623d22db24,client_e629fa6598,10,0,1,20,down,1,0.175603
17690,content_c268b1716236,client_e629fa6598,3,0,2,20,down,1,0.178301
23523,content_38b9529bf1b6,client_e629fa6598,36,0,2,22,down,1,0.180157
4618,content_1815d425a3a0,client_e629fa6598,11,0,2,20,down,1,0.180660
7991,content_ce296f93e007,client_e629fa6598,4,0,1,22,down,1,0.181678
16610,content_6248c9728c62,client_e629fa6598,8,0,4,20,down,1,0.184840


MOST INFLUENTIAL FEATURES


,feature,coefficient,absolute_coefficient
8,numeric__impressions_last_30d,-35.287714,35.287714
11,numeric__impressions_prev_30d,29.303749,29.303749
2,numeric__sessions_90d,1.418420,1.418420
0,numeric__impressions_90d,1.415256,1.415256
12,numeric__clicks_prev_30d,1.128250,1.128250
9,numeric__clicks_last_30d,-1.066579,1.066579
3,numeric__users_90d,-1.024830,1.024830
28,categorical__main_intent_navigational,-0.522831,0.522831
6,numeric__days_with_impressions,0.490799,0.490799
24,categorical__content_type_feedly article,-0.431369,0.431369



INTERPRETATION:

The error tables show where the model's ranking disagrees with the
observed decline label.

False positives may represent pages that look risky according to
their available historical signals but do not decline in the observed
label period.

False negatives may represent pages whose available features look
healthy even though they are labelled as declining.

The coefficient table provides a directional view of which features
the Logistic Regression model relies on most. These relationships
should be treated as associations in this dataset, not causal effects.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.